### Insurance Website Chatbot (vLLM backend + Tools + Tests)

This notebook is intentionally split into **three parts**:

1) **Model backend (vLLM server)**
- Downloads/loads the Hugging Face model: https://huggingface.co/Qwen/Qwen3-30B-A3B-Instruct-2507-FP8
- Starts an OpenAI-compatible API server at `http://localhost:8000/v1`

2) **Tools + Agent (LangGraph ReAct)**
- Defines simple insurance website tools (OSAGO, travel, documents, claim steps, callback)
- Creates a strict ReAct agent that **must not guess missing arguments**

3) **Tests (anti-hallucination demo)**
- Sends user-like queries and checks that the agent asks for missing fields

Notes:
- In Colab, `localhost` refers to the Colab VM. If you start vLLM inside this notebook, `http://localhost:8000/v1` works.
- If you run vLLM elsewhere, set `VLLM_BASE_URL` to that external endpoint.
- This notebook does not use OpenAI servers; it only uses an OpenAI-compatible HTTP interface.


In [1]:
# 1) Install dependencies (Colab/Linux recommended)
# vLLM runs reliably on Linux (Colab). On Windows, use WSL2 or run vLLM on a Linux machine.
import sys

!{sys.executable} -m pip install -U -q vllm langgraph langchain-openai langchain-core pydantic requests psutil nest_asyncio

import nest_asyncio
nest_asyncio.apply()

print("[INFO] Dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 114.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1

In [6]:
# 2) Start vLLM backend server (downloads + loads the Hugging Face model)
# Model: https://huggingface.co/Qwen/Qwen3-30B-A3B-Instruct-2507-FP8

import os
import sys
import time
import subprocess
import requests
import psutil
import gc

# Fix for potential protobuf errors
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

MODEL_ID = "Qwen/Qwen3-30B-A3B-Instruct-2507-FP8"
PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"

# If Hugging Face requires auth for you, set this:
# os.environ["HF_TOKEN"] = "<your_hf_token>"

print("[INFO] Cleaning up old server processes...")
for proc in psutil.process_iter(["pid", "name"]):
    try:
        for conn in proc.connections(kind="inet"):
            if conn.laddr and conn.laddr.port == PORT:
                print(f"[INFO] Killing PID={proc.pid} name={proc.name()} on port {PORT}")
                proc.kill()
    except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
        pass

# Optional: clear CUDA cache
try:
    import torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("[INFO] CUDA cache cleared.")
except Exception:
    pass

time.sleep(2)

# IMPORTANT: enable tool calling support in vLLM (required for agent tools)
cmd = [
    sys.executable,
    "-m",
    "vllm.entrypoints.openai.api_server",
    "--model",
    MODEL_ID,
    "--served-model-name",
    MODEL_ID,
    "--trust-remote-code",
    "--dtype",
    "auto",
    "--max-model-len",
    "8192",
    "--gpu-memory-utilization",
    "0.90",
    "--host",
    "0.0.0.0",
    "--port",
    str(PORT),
    "--enable-auto-tool-choice",
    "--tool-call-parser",
    "hermes",
]

print("[INFO] Starting vLLM server...")
print("[INFO] Command:", " ".join(cmd))

log_path = "vllm_server.log"
log_file = open(log_path, "w", encoding="utf-8")
process = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)

print(f"[INFO] Server PID={process.pid}")
print("[INFO] Waiting for /models to be available (can take a few minutes)...")

start = time.time()
while True:
    if process.poll() is not None:
        print("[ERROR] vLLM process exited early.")
        print("[ERROR] Check vllm_server.log for details.")
        break

    try:
        r = requests.get(f"{BASE_URL}/models", timeout=5)
        if r.ok:
            print("[INFO] vLLM is online.")
            print("[INFO] BASE_URL:", BASE_URL)
            print("[INFO] /models (short):", r.text[:300])
            break
    except requests.exceptions.RequestException:
        pass

    if time.time() - start > 900:
        print("[ERROR] Timeout waiting for vLLM server.")
        print("[ERROR] Check vllm_server.log")
        process.kill()
        break

    time.sleep(5)
    print(".", end="", flush=True)

[INFO] Cleaning up old server processes...
[INFO] CUDA cache cleared.
[INFO] Starting vLLM server...
[INFO] Command: /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-30B-A3B-Instruct-2507-FP8 --served-model-name Qwen/Qwen3-30B-A3B-Instruct-2507-FP8 --trust-remote-code --dtype auto --max-model-len 8192 --gpu-memory-utilization 0.90 --host 0.0.0.0 --port 8000 --enable-auto-tool-choice --tool-call-parser hermes
[INFO] Server PID=2840
[INFO] Waiting for /models to be available (can take a few minutes)...
.............................................................[INFO] vLLM is online.
[INFO] BASE_URL: http://localhost:8000/v1
[INFO] /models (short): {"object":"list","data":[{"id":"Qwen/Qwen3-30B-A3B-Instruct-2507-FP8","object":"model","created":1770877021,"owned_by":"vllm","root":"Qwen/Qwen3-30B-A3B-Instruct-2507-FP8","parent":null,"max_model_len":8192,"permission":[{"id":"modelperm-af908cb60f0ecef1","object":"model_permission","created":177087


In [3]:
# 3) Tools + Strict Agent (LangGraph ReAct)

from __future__ import annotations

from typing import Dict, List
from datetime import date
import re

from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

# Use the vLLM server from Cell 2
MODEL_ID = "Qwen/Qwen3-30B-A3B-Instruct-2507-FP8"
BASE_URL = "http://localhost:8000/v1"
API_KEY = "EMPTY"

SYSTEM_PROMPT = (
    "You are a strict insurance website assistant.\n\n"
    "Rules:\n"
    "1) Never invent missing parameters.\n"
    "2) Ask short follow-up questions for any missing required field.\n"
    "3) Call tools only when all required fields are provided.\n"
    "4) Keep answers clear and short.\n"
    "5) Currency is RUB.\n"
)

CALLBACKS: List[Dict[str, str]] = []


def _valid_phone(phone: str) -> bool:
    return bool(re.fullmatch(r"[+]?\d{10,15}", phone.replace(" ", "")))


class OSAGOArgs(BaseModel):
    car_power_hp: int = Field(..., description="Car power in horsepower (positive int)")
    driver_age: int = Field(..., description="Driver age in years (>= 18)")
    experience_years: int = Field(..., description="Driving experience in years (>= 0)")
    region: str = Field(..., description="Region name/code (e.g., Moscow, SPB)")


@tool(args_schema=OSAGOArgs)
def calculate_osago(car_power_hp: int, driver_age: int, experience_years: int, region: str) -> str:
    """Estimate OSAGO (simplified)."""
    try:
        if car_power_hp <= 0:
            return "[ERROR] car_power_hp must be positive."
        if driver_age < 18:
            return "[ERROR] driver_age must be >= 18."
        if experience_years < 0:
            return "[ERROR] experience_years must be >= 0."
        if not region.strip():
            return "[ERROR] region is required."

        base = 5000.0
        power_factor = 1.0 if car_power_hp <= 100 else 1.2 if car_power_hp <= 150 else 1.4
        age_factor = 1.25 if driver_age < 23 else 1.0
        exp_factor = 1.3 if experience_years < 3 else 1.0
        region_factor = 1.2 if region.lower() in ["moscow", "msk", "saint petersburg", "spb"] else 1.0

        price = base * power_factor * age_factor * exp_factor * region_factor
        return f"[TOOL] OSAGO estimate: {price:.0f} RUB per year."
    except Exception as e:
        return f"[ERROR] calculate_osago failed: {e}"


class TravelArgs(BaseModel):
    days: int = Field(..., description="Trip duration in days (positive int)")
    country: str = Field(..., description="Destination country")
    sports: bool = Field(False, description="Whether sports/extreme coverage is needed")


@tool(args_schema=TravelArgs)
def calculate_travel_insurance(days: int, country: str, sports: bool = False) -> str:
    """Estimate travel insurance (simplified)."""
    try:
        if days <= 0:
            return "[ERROR] days must be positive."
        if not country.strip():
            return "[ERROR] country is required."

        daily = 120.0
        sports_factor = 1.8 if sports else 1.0
        price = days * daily * sports_factor
        return f"[TOOL] Travel insurance estimate: {price:.0f} RUB total."
    except Exception as e:
        return f"[ERROR] calculate_travel_insurance failed: {e}"


class DocsArgs(BaseModel):
    product: str = Field(..., description="Product: osago, casco, travel, property")


@tool(args_schema=DocsArgs)
def get_required_documents(product: str) -> str:
    """Return required documents for a product."""
    try:
        p = product.strip().lower()
        docs = {
            "osago": ["Passport", "Driver license", "Vehicle registration (STS)", "VIN/plate data"],
            "casco": ["Passport", "Driver license", "Vehicle registration (STS)", "Vehicle photos"],
            "travel": ["Passport", "Travel dates", "Destination country"],
            "property": ["Passport", "Property address", "Ownership proof"],
        }
        if p not in docs:
            return "[ERROR] Unknown product. Use: osago, casco, travel, property."
        return "[TOOL] Required documents: " + ", ".join(docs[p])
    except Exception as e:
        return f"[ERROR] get_required_documents failed: {e}"


class ClaimArgs(BaseModel):
    claim_type: str = Field(..., description="Type: car_accident, property_damage, medical_travel")


@tool(args_schema=ClaimArgs)
def check_claim_steps(claim_type: str) -> str:
    """Return short steps for a claim."""
    try:
        ct = claim_type.strip().lower()
        if ct == "car_accident":
            return "[TOOL] Steps: 1) Ensure safety 2) Call emergency if needed 3) Take photos 4) Submit claim with incident details."
        if ct == "property_damage":
            return "[TOOL] Steps: 1) Photo/video evidence 2) Minimize further damage 3) Contact support 4) Submit documents."
        if ct == "medical_travel":
            return "[TOOL] Steps: 1) Contact assistance line 2) Keep all invoices 3) Submit medical docs and receipts."
        return "[ERROR] Unknown claim_type. Use: car_accident, property_damage, medical_travel."
    except Exception as e:
        return f"[ERROR] check_claim_steps failed: {e}"


class CallbackArgs(BaseModel):
    full_name: str = Field(..., description="Customer full name")
    phone: str = Field(..., description="Phone number: 10-15 digits, optional +")
    topic: str = Field(..., description="Callback topic")


@tool(args_schema=CallbackArgs)
def create_callback_request(full_name: str, phone: str, topic: str) -> str:
    """Create a callback request."""
    try:
        if not full_name.strip():
            return "[ERROR] full_name is required."
        if not _valid_phone(phone):
            return "[ERROR] Invalid phone format. Use 10-15 digits, optional +."
        if not topic.strip():
            return "[ERROR] topic is required."

        req_id = len(CALLBACKS) + 1
        CALLBACKS.append(
            {
                "id": str(req_id),
                "full_name": full_name.strip(),
                "phone": phone.strip(),
                "topic": topic.strip(),
                "created": date.today().isoformat(),
            }
        )
        return f"[TOOL] Callback created. request_id={req_id}."
    except Exception as e:
        return f"[ERROR] create_callback_request failed: {e}"


llm = ChatOpenAI(
    model=MODEL_ID,
    base_url=BASE_URL,
    api_key=API_KEY,
    temperature=0.0,
)

tools = [
    calculate_osago,
    calculate_travel_insurance,
    get_required_documents,
    check_claim_steps,
    create_callback_request,
]

# Create LangGraph ReAct agent (uses tool-calling)
agent = create_react_agent(llm, tools)

print("[INFO] Tools exposed to the model:")
for t in tools:
    schema = getattr(t, "args_schema", None)
    fields = list(getattr(schema, "model_fields", {}).keys()) if schema else []
    print(f"- {t.name}({', '.join(fields)})")

print("[INFO] Agent created.")


[INFO] Tools exposed to the model:
- calculate_osago(car_power_hp, driver_age, experience_years, region)
- calculate_travel_insurance(days, country, sports)
- get_required_documents(product)
- check_claim_steps(claim_type)
- create_callback_request(full_name, phone, topic)
[INFO] Agent created.


/tmp/ipython-input-3353935186.py:181: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


In [7]:
# 4) Tests: demonstrate anti-hallucination (missing params -> agent asks)

from typing import List, Tuple


def run_turn(history: List[Tuple[str, str]], user_text: str) -> str:
    history.append(("user", user_text))
    out = agent.invoke({"messages": history})
    answer = out["messages"][-1].content
    history.append(("assistant", answer))
    return answer


history: List[Tuple[str, str]] = [("system", SYSTEM_PROMPT)]

print("--- Test 1: OSAGO missing fields (should ask) ---")
q1 = "I want to calculate OSAGO for my car."
print("[USER]", q1)
print("[ASSISTANT]", run_turn(history, q1))
print()

print("--- Provide OSAGO params (should call tool) ---")
q2 = "Power 140 hp, age 29, experience 7 years, region Moscow."
print("[USER]", q2)
print("[ASSISTANT]", run_turn(history, q2))
print()

print("--- Test 2: Callback missing fields (should ask) ---")
q3 = "Please call me back."
print("[USER]", q3)
print("[ASSISTANT]", run_turn(history, q3))
print()

print("--- Provide callback params (should call tool) ---")
q4 = "My name is Ivan Petrov. Phone +79991234567. Topic: OSAGO renewal."
print("[USER]", q4)
print("[ASSISTANT]", run_turn(history, q4))
print()

print("--- Test 3: Travel insurance missing fields (should ask) ---")
q5 = "How much is travel insurance?"
print("[USER]", q5)
print("[ASSISTANT]", run_turn(history, q5))
print()

print("--- Provide travel params (should call tool) ---")
q6 = "10 days in Turkey, no sports."
print("[USER]", q6)
print("[ASSISTANT]", run_turn(history, q6))

--- Test 1: OSAGO missing fields (should ask) ---
[USER] I want to calculate OSAGO for my car.
[ASSISTANT] Please provide the following details:
- Car power in horsepower (e.g., 120)
- Driver age in years (minimum 18)
- Driving experience in years (minimum 0)
- Region (e.g., Moscow, SPB)

--- Provide OSAGO params (should call tool) ---
[USER] Power 140 hp, age 29, experience 7 years, region Moscow.
[ASSISTANT] The estimated annual cost for OSAGO is 7,200 RUB.

--- Test 2: Callback missing fields (should ask) ---
[USER] Please call me back.
[ASSISTANT] Please provide your full name and phone number for the callback request.

--- Provide callback params (should call tool) ---
[USER] My name is Ivan Petrov. Phone +79991234567. Topic: OSAGO renewal.
[ASSISTANT] Your callback request has been created (ID: 1). A representative will contact you shortly.

--- Test 3: Travel insurance missing fields (should ask) ---
[USER] How much is travel insurance?
[ASSISTANT] Please provide the following d

In [8]:
# Check vLLM server log
print("--- vllm_server.log ---")
!cat vllm_server.log

--- vllm_server.log ---
2026-02-12 06:12:01.538697: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-12 06:12:01.556948: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770876721.580820    2840 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770876721.588070    2840 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770876721.605711    2840 computation_placer.cc:177] computation placer already registered. Please check li